# Principal Component Analysis (PCA)

**Dataset:** Digits (sklearn built-in)  
**Task:** Dimensionality reduction — compress 64 pixel features into fewer dimensions

---

## Overview

**PCA** finds the directions of **maximum variance** in high-dimensional data and projects the data onto a lower-dimensional subspace. It is one of the most widely used dimensionality reduction techniques.

### Algorithm

1. **Center** the data: $\tilde{X} = X - \bar{X}$
2. Compute the **covariance matrix**: $\Sigma = \frac{1}{n-1} \tilde{X}^\top \tilde{X}$
3. Compute **eigenvectors and eigenvalues** of $\Sigma$: $\Sigma \mathbf{v}_i = \lambda_i \mathbf{v}_i$
4. Sort by eigenvalue (descending) — the $i$-th eigenvector is the $i$-th **principal component (PC)**
5. **Project** data onto the top $k$ PCs: $Z = \tilde{X} W_k$ where $W_k$ contains the top $k$ eigenvectors

### Explained Variance

The fraction of total variance explained by $k$ components:

$$\text{EVR}_k = \frac{\sum_{i=1}^k \lambda_i}{\sum_{i=1}^p \lambda_i}$$

This tells us how much information is retained after projection.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, load_iris
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

np.random.seed(42)
plt.rcParams['font.size'] = 12

## 2. PCA from Scratch

In [ ]:
class PCAScratch:
    """PCA via eigendecomposition of the covariance matrix."""

    def __init__(self, n_components):
        self.n_components = n_components

    def fit(self, X):
        self.mean_ = X.mean(axis=0)
        X_centered = X - self.mean_

        # Covariance matrix and eigendecomposition
        cov = np.cov(X_centered, rowvar=False)
        eigenvalues, eigenvectors = np.linalg.eigh(cov)

        # Sort descending
        idx = np.argsort(eigenvalues)[::-1]
        self.eigenvalues_ = eigenvalues[idx]
        self.components_ = eigenvectors[:, idx].T  # rows = principal components

        total_var = self.eigenvalues_.sum()
        self.explained_variance_ratio_ = self.eigenvalues_ / total_var

        return self

    def transform(self, X):
        return (X - self.mean_) @ self.components_[:self.n_components].T

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    def inverse_transform(self, Z):
        return Z @ self.components_[:self.n_components] + self.mean_

## 3. Load the Digits Dataset

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target

print(f"Original shape: {X.shape}  (1797 samples × 64 pixel features)")
print(f"We want to visualize and compress this into much fewer dimensions.")

## 4. Explained Variance

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca_full = PCAScratch(n_components=64)
pca_full.fit(X_scaled)

cumulative_evr = np.cumsum(pca_full.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(range(1, 21), pca_full.explained_variance_ratio_[:20], color='steelblue', alpha=0.8)
axes[0].set_xlabel('Principal Component'); axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Individual Explained Variance (first 20 PCs)')
axes[0].grid(axis='y', alpha=0.3)

axes[1].plot(range(1, 65), cumulative_evr, 'o-', color='steelblue', markersize=3)
for thresh, color in [(0.80, 'red'), (0.90, 'orange'), (0.95, 'green')]:
    k = np.searchsorted(cumulative_evr, thresh) + 1
    axes[1].axhline(thresh, color=color, linestyle='--', alpha=0.7, label=f'{int(thresh*100)}%: {k} PCs')
axes[1].set_xlabel('Number of Components'); axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('PCA Explained Variance — Digits Dataset', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. 2D Visualization

In [ ]:
pca_2d = PCAScratch(n_components=2)
X_2d = pca_2d.fit_transform(X_scaled)

plt.figure(figsize=(9, 7))
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y, cmap='tab10', alpha=0.7, s=20)
plt.colorbar(scatter, label='Digit')
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.title('Digits Dataset — Projected onto First 2 Principal Components')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Image Reconstruction

PCA compression: reduce 64 dimensions to $k$, then reconstruct. More components = better reconstruction.

In [ ]:
n_components_list = [1, 5, 10, 20, 40, 64]
sample_idx = np.where(y == 4)[0][0]  # pick one '4' digit
x_original = X_scaled[sample_idx:sample_idx+1]

fig, axes = plt.subplots(1, len(n_components_list) + 1, figsize=(16, 3))

# Original
axes[0].imshow(scaler.inverse_transform(x_original).reshape(8,8), cmap='gray_r')
axes[0].set_title('Original'); axes[0].axis('off')

for ax, k in zip(axes[1:], n_components_list):
    pca_k = PCAScratch(n_components=k)
    pca_k.fit(X_scaled)
    z = pca_k.transform(x_original)
    x_rec = pca_k.inverse_transform(z)
    evr = pca_full.explained_variance_ratio_[:k].sum()
    ax.imshow(scaler.inverse_transform(x_rec).reshape(8,8), cmap='gray_r')
    ax.set_title(f'{k} PCs\n({evr*100:.0f}% var)', fontsize=9)
    ax.axis('off')

plt.suptitle('Image Reconstruction with Varying Numbers of PCs', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. PCA for Preprocessing: Effect on Classification Accuracy

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

k_vals = [2, 5, 10, 20, 30, 40, 64]
accs = []

for k in k_vals:
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=k)),
        ('clf', LogisticRegression(max_iter=1000))
    ])
    pipe.fit(X_train, y_train)
    accs.append(pipe.score(X_test, y_test))

plt.figure(figsize=(8, 5))
plt.plot(k_vals, accs, 'o-', color='steelblue', lw=2)
plt.xlabel('Number of PCA Components')
plt.ylabel('Test Accuracy')
plt.title('Logistic Regression Accuracy vs. PCA Dimensions')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("k | Accuracy")
for k, a in zip(k_vals, accs):
    print(f"{k:3d} | {a:.4f}")

## 8. Visualize Principal Components

In [ ]:
pca_vis = PCAScratch(n_components=16)
pca_vis.fit(X_scaled)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
axes = axes.ravel()

for i in range(16):
    pc = pca_vis.components_[i].reshape(8, 8)
    axes[i].imshow(pc, cmap='RdBu_r')
    axes[i].set_title(f'PC {i+1}\n({pca_vis.explained_variance_ratio_[i]*100:.1f}%)', fontsize=8)
    axes[i].axis('off')

plt.suptitle('First 16 Principal Components (Eigendigits)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Summary

### Key Takeaways

- PCA finds the **directions of maximum variance** using eigendecomposition of the covariance matrix.
- The **scree plot** and **cumulative explained variance** guide the choice of $k$ components to retain.
- PCA can dramatically **compress** data: ~20 components capture ~90% of variance in the 64-dimensional Digits dataset.
- The 2D projection shows that many digits are **linearly separable** in PCA space.
- **Reconstruction** quality improves smoothly with $k$ — even 10 PCs produce recognizable digits.
- PCA is most effective as **preprocessing** for downstream tasks when the original features are correlated.
- PCA is sensitive to feature **scale** — always standardize first.